# Task 4: Outer Vector Add (Block)

Bot task id: `04_outer_vector_add_block`, deadline 06.12.2026 00:00 MSK. [Open in Colab](https://colab.research.google.com/github/PE51K/itmo-ai-talent-hub-efficient-models-course/blob/main/hw3/04_outer_vector_add_block.ipynb)

## Условие

Внешняя сумма двух векторов с разбиением на блоки:

$$z[i, j] = x[j] + y[i], \quad i = 0 \dots N_1-1,\ j = 0 \dots N_0-1$$

**Требования**
- сетка двумерная: `(pid1, pid0)` — строки и столбцы;
- размеры блоков `B0 < N0`, `B1 < N1`, поэтому нужно маскировать хвосты;
- использовать `tl.arange`, `tl.load`, `tl.store`;
- решение должно работать на GPU (CUDA).

**Подсказка**
```python
pid0 = tl.program_id(0)
pid1 = tl.program_id(1)
offs0 = pid0 * B0 + tl.arange(0, B0)
offs1 = pid1 * B1 + tl.arange(0, B1)
```
- маски: `mask0 = offs0 < N0`, `mask1 = offs1 < N1`;
- broadcast: `y[:, None] + x[None, :]`;
- смещения `offs = offs1[:, None] * N0 + offs0[None, :]`, запись с маской `mask1[:, None] & mask0[None, :]`.

Проверка: `N0 = 100`, `N1 = 90`.

## Setup

Runtime -> Change runtime type -> T4 GPU.

In [ ]:
import importlib.util

import torch
import triton

assert torch.cuda.is_available(), "no GPU in this runtime: Runtime -> Change runtime type -> T4 GPU"
print(f"torch {torch.__version__}, triton {triton.__version__}, {torch.cuda.get_device_name()}")

## Solution

`%%writefile` saves the cell to `04_outer_vector_add_block.py`, the file to submit. Run it after every edit, then the check.

In [ ]:
%%writefile 04_outer_vector_add_block.py
import torch
import triton
import triton.language as tl


@triton.jit
def _outer_add_block_kernel(x_ptr, y_ptr, z_ptr, N0, N1, B0: tl.constexpr, B1: tl.constexpr):
    # YOUR CODE HERE
    pass


def add_vec_block_triton(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    # YOUR CODE HERE
    raise NotImplementedError

## Check

Compares with PyTorch on shapes with tails and checks some of the requirements. These are my tests, not the bot's: passing them doesn't guarantee the bot accepts the solution.

In [ ]:
PATH = "04_outer_vector_add_block.py"
spec = importlib.util.spec_from_file_location("sol", PATH)
sol = importlib.util.module_from_spec(spec)
spec.loader.exec_module(sol)
SRC = open(PATH).read()

x, y = torch.randn(100, device="cuda"), torch.randn(90, device="cuda")
torch.testing.assert_close(sol.add_vec_block_triton(x, y), y[:, None] + x[None, :])

torch.cuda.synchronize()
print("PASS")

## Download

In [ ]:
from google.colab import files

files.download("04_outer_vector_add_block.py")